# 00 — setup and the per-profile asset cache

Run this once per machine or Kaggle session. It sets the **only** knob (`ACTIVE`), prints what the
box can do, resolves the Qwen3.5-4B weights and the wheelhouse from the profile cache (attached
Kaggle dataset → local cache → verified local download → network), asserts the DeltaNet fast path,
and prints the datasets to attach next time so the next run downloads nothing.

Everything lands inside `models/<profile>/`, `data/<profile>/`, `runs/<profile>/`: two profiles never
share an adapter, a checkpoint, a log or a cache.

In [ ]:
ACTIVE = "local_4050"          # local_4050 | local_4090 | kaggle_t4x2 | pro6000 | tpu_v5e8
import os, sys
from pathlib import Path

os.environ["RTP_PROFILE"] = ACTIVE
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("profile:", ACTIVE, "| repo:", ROOT)

In [ ]:
import platform, subprocess

KAGGLE = Path("/kaggle/input").is_dir()
print("kaggle:", KAGGLE, "| python:", platform.python_version(), "|", platform.platform())
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True)
print(gpu.stdout.strip() or "(no nvidia-smi: this box will not train)")

In [ ]:
# install-branch recommendation (Newton floor, Warp pin, QLoRA fit); exits 0 with warnings only
r = subprocess.run([sys.executable, "ops/preflight.py"], capture_output=True, text=True)
print(r.stdout[-2000:])

In [ ]:
from rtp.profiles import detect_caps, get_profile, resolve_runtime

prof = get_profile(ACTIVE)
prof.ensure_dirs()
caps = detect_caps()
attn, dtype, notes = resolve_runtime(prof, caps)
print("paths:", prof.paths.as_dict())
print("config hash:", prof.config_hash(), "| effective batch:", prof.effective_batch)
print("attention:", attn, "| dtype:", dtype)
for note in notes:
    print(" -", note)

In [ ]:
from rtp import assetcache as ac
from rtp.profiles import record_stage, stage_done

MODEL_REV = "851bf6e8"       # HF commit of the verified download (manifest hashes pin the bytes)
LOCAL_SEED = Path.home() / "models" / "Qwen3.5-4B"
spec = ac.model_spec("Qwen/Qwen3.5-4B", revision=MODEL_REV, name="qwen35-4b")
dest = ac.asset_dir(spec, prof)

def download(d):
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3.5-4B", revision=MODEL_REV, local_dir=d)

if not ac.verify_manifest(dest, spec):
    print("cache hit:", dest)
    model_dir = dest
elif KAGGLE:
    model_dir = ac.resolve(spec, prof, attached="/kaggle/input", verify="size")
elif LOCAL_SEED.is_dir():
    print("seeding the cache from the verified local download (hardlinks, no network)")
    model_dir = ac.seed_from_dir(LOCAL_SEED, spec, prof)
else:
    model_dir = ac.resolve(spec, prof, fetch=download, verify="size")

print("model:", model_dir, "| files:", len(list(ac.iter_files(model_dir))))
print("setup recorded before this cell:", stage_done(prof, "setup", prof.config_hash()))
record_stage(prof, "setup", status="done", config_hash=prof.config_hash())

In [ ]:
# the wheelhouse is what makes a Kaggle run install offline: wheels built once for this profile's tags
tags = ac.env_tags()
wheels = ac.wheelhouse_spec(python=tags["python"], torch=tags.get("torch", "?"),
                            cuda=tags.get("cuda", "?"), sm=tags.get("sm", "?"))
mounted = Path("/kaggle/input") / wheels.slug
if mounted.is_dir():
    print("wheelhouse attached:", mounted)
    print("install offline with: ops/install.sh --profile", ACTIVE, "--wheelhouse", mounted)
else:
    print("no wheelhouse attached; expected dataset:", wheels.slug)
    print("build it once for these tags:", tags)

In [ ]:
# fast path: the DeltaNet layers must call causal_conv1d + fla, not the pure-torch fallback
if caps.cuda:
    r = subprocess.run([sys.executable, "ops/g2_fastpath.py", "--model", str(model_dir),
                        "--out", str(prof.paths.reports / "g2_fastpath.json")],
                       capture_output=True, text=True)
    print(r.stdout[-1500:])
    assert r.returncode == 0, "DeltaNet fast path is not bound; see the output above"
else:
    print("no CUDA: skipping the fast-path assertion (those kernels are GPU-only)")

In [ ]:
EXPORT_TO_KAGGLE = False     # publishing needs Kaggle CLI auth (gate G-11); off by default

print(ac.attach_hint(spec))
print("the wheelhouse dataset:", ac.wheelhouse_spec(
    python=ac.env_tags()["python"], torch=ac.env_tags().get("torch", "?"),
    cuda=ac.env_tags().get("cuda", "?"), sm=ac.env_tags().get("sm", "?")).slug)
if EXPORT_TO_KAGGLE:
    try:
        ac.export_private(dest, spec, owner=ac.kaggle_owner())
    except ac.ExportNotConfigured as exc:
        print("export skipped:", exc)

Next: the stage notebooks (`10_sft`, `20_grpo`, `30_sim_eval`, `40_teacher`) are added as their
stages land; each starts with the same `ACTIVE` cell and calls `stage_done()` before doing work, so
"run all" is safe to repeat.

Anything this notebook wrote is inside the profile tree above. Nothing outside it is touched.